# Level 2 — Aggregate one country (no Coiled)

Tests `_aggregate_partition` directly — one country, one hazard.
Scored zarr must already exist on GCS (run Level 1 first).

In [ ]:
import sys
sys.path.insert(0, "/home/stefan/CRS/CRS.ZarrPipelines/")
import pandas as pd
import matplotlib.pyplot as plt
from app.domain.gadm_aggregations import _aggregate_partition, load_gadm

## Run aggregation for one country

Change `gid0` and `hazard_code` to test different hazards.
CF/RF route through custom RP-weighted aggregation for scales 5/10; plain zonal stats for 100.
All other hazards use plain xvec zonal stats.

In [ ]:
df = _aggregate_partition(
    gid0="ETH",
    scored_zarr_path="gs://crs_climate_data_public/production_test/hazard_scores/HS.zarr",
    hazard_code="HS",
    gadm_level=1,
)
print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
print(f"Scoring scales: {sorted(df['scoring'].unique())}")
df.head()

## Inspect result

In [ ]:
df

## Plot score_mean distribution per scoring scale

In [ ]:
# Filter to one scenario/time/statistic to keep the plot clean
scales = sorted(df["scoring"].unique())
n = len(scales)
fig, axes = plt.subplots(1, n, figsize=(5 * n, 4), sharey=True)
if n == 1:
    axes = [axes]

subset = df[(df["statistic"] == "mean") & (df["scenario"] == "RCP45") & (df["time"] == "Cc")]

for ax, scale in zip(axes, scales):
    group = subset[subset["scoring"] == scale]
    group["score_mean"].dropna().hist(ax=ax, bins=20, edgecolor="white")
    label = f"{scale}-point" if str(scale) != "100" else "0-100 min-max"
    ax.set_title(f"HS — {label}")
    ax.set_xlabel("score_mean")
    ax.set_ylabel("province count")

plt.suptitle("HS score_mean per province (ETH, RCP45, mean, Cc)")
plt.tight_layout()
plt.show()

## Inspect GADM boundaries

In [ ]:
gadm = load_gadm(1, iso3="ETH")
print(f"Provinces: {len(gadm)}")
print(gadm[["GID_1", "GID_0"]].head())
gadm.plot(figsize=(6, 4))
plt.title("ETH provinces (GADM adm1)")
plt.show()